# Notebook 4: RLHF — Reward Model + PPO

## What we're doing
Full RLHF pipeline in two phases:
- **Phase 1**: Train a reward model (RM) on labelled phishing URL analysis pairs
- **Phase 2**: Use PPO to fine-tune the policy using the RM as the reward signal

## Task
**Phishing URL explanation quality** — the model explains whether a URL is phishing.
The RM learns to score explanation quality. PPO then trains the policy to maximize RM scores.

## Dataset
- Primary: `pirocheto/phishing-url` (HuggingFace)
- Fallback: synthetic phishing/legit URL pairs with good/bad explanations

## RLHF pipeline
```
Phase 1 — Reward Model:
  Input:  (prompt, response) → scalar score
  Loss:   MSE(predicted_score, human_label)
  Output: RM that knows what a good explanation looks like

Phase 2 — PPO:
  For each step:
    1. Policy generates response y ~ π_θ(·|x)
    2. RM scores it: r = RM(x, y)
    3. KL penalty computed: kl = KL(π_θ(y|x) ∥ π_ref(y|x))
    4. Reward: r_adjusted = r - β * kl
    5. PPO clip update on policy
```

## CPU config
- RM: small classifier head on Qwen2.5-0.5B, trained for 1 epoch
- PPO: mini rollout buffer, 1 PPO epoch per batch, max_new_tokens=64
- expect ~90–180 min total on CPU

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'trl>=0.8', 'datasets', 'accelerate', 'torch'
])
print('Dependencies ready.')

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset
from transformers import TrainingArguments
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead

from data.loader import load_rlhf_dataset, train_val_split
from shared.model_utils import (
    load_model_and_tokenizer, load_saved_model,
    generate_response, LossTracker, save_model, MODEL_ID
)

os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.manual_seed(42)
print(f'PyTorch {torch.__version__} | cpu')

In [ ]:
# ── Cell 3: Load phishing dataset ────────────────────────────────────────────
MAX_SAMPLES = 120

rm_data, ppo_prompts = load_rlhf_dataset(max_samples=MAX_SAMPLES)
rm_train, rm_val = train_val_split(rm_data, val_ratio=0.1)

print(f'RM training samples: {len(rm_train)} | Val: {len(rm_val)}')
print(f'PPO prompts: {len(ppo_prompts)}')
print('\n--- RM sample ---')
s = rm_train[0]
print(f'prompt:   {s["prompt"][:100]}...')
print(f'response: {s["response"][:100]}...')
print(f'label:    {s["label"]} (1=good, 0=bad)')

In [ ]:
# ── Cell 4: Phase 1 — Train Reward Model ─────────────────────────────────────
#
# The RM is Qwen2.5-0.5B with the LM head replaced by a linear layer → scalar score.
# We train it to output high scores for good explanations, low for bad.
#
# Architecture:
#   Qwen transformer backbone (frozen or fine-tuned)
#   → take last token hidden state (shape: [batch, hidden_size])
#   → linear layer → scalar score
#
# Loss: Binary Cross Entropy (treat as quality classification)

print('=== Phase 1: Training Reward Model ===')
print('Loading RM backbone (AutoModelForSequenceClassification)...')

from shared.model_utils import load_model_and_tokenizer
rm_model, rm_tokenizer = load_model_and_tokenizer(for_reward_model=True)

print(f'RM loaded: {sum(p.numel() for p in rm_model.parameters())/1e6:.0f}M params')
print(f'RM output head: {rm_model.config.num_labels} label(s) (scalar reward)')

In [ ]:
# ── Cell 5: Tokenize RM training data ────────────────────────────────────────
MAX_RM_LEN = 128

def tokenize_rm_sample(sample, tokenizer, max_len=MAX_RM_LEN):
    """Tokenize prompt+response concatenation for the reward model."""
    text = sample['prompt'] + ' ' + sample['response']
    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        padding='max_length',
        return_tensors='pt'
    )
    return {
        'input_ids':      enc['input_ids'].squeeze(0),
        'attention_mask': enc['attention_mask'].squeeze(0),
        'labels':         torch.tensor([float(sample['label'])]),
    }

rm_train_tok = [tokenize_rm_sample(s, rm_tokenizer) for s in rm_train]
rm_val_tok   = [tokenize_rm_sample(s, rm_tokenizer) for s in rm_val]
print(f'Tokenized: {len(rm_train_tok)} train, {len(rm_val_tok)} val')
print(f'Token shape: {rm_train_tok[0]["input_ids"].shape}')

In [ ]:
# ── Cell 6: RM training loop ──────────────────────────────────────────────────
from torch.optim import AdamW

RM_EPOCHS = 1
RM_LR     = 1e-5

optimizer = AdamW(rm_model.parameters(), lr=RM_LR)
loss_fn   = nn.BCEWithLogitsLoss()   # logit → sigmoid → BCE

rm_model.train()
rm_tracker = LossTracker()

print(f'Training RM for {RM_EPOCHS} epoch(s), {len(rm_train_tok)} samples...')

for epoch in range(RM_EPOCHS):
    total_loss = 0
    for step, sample in enumerate(rm_train_tok):
        optimizer.zero_grad()

        input_ids      = sample['input_ids'].unsqueeze(0)      # (1, L)
        attention_mask = sample['attention_mask'].unsqueeze(0)  # (1, L)
        label          = sample['labels']                       # (1,)

        outputs = rm_model(input_ids=input_ids, attention_mask=attention_mask)
        logit   = outputs.logits.squeeze(-1)  # (1,)

        loss = loss_fn(logit, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        rm_tracker.record(step, loss.item())

        if step % 20 == 0:
            print(f'  Epoch {epoch+1} Step {step:3d} | loss={loss.item():.4f}')

    print(f'Epoch {epoch+1} done | avg loss={total_loss/len(rm_train_tok):.4f}')

# RM validation accuracy
rm_model.eval()
correct = 0
with torch.no_grad():
    for sample in rm_val_tok:
        out  = rm_model(sample['input_ids'].unsqueeze(0), sample['attention_mask'].unsqueeze(0))
        pred = (out.logits.squeeze() > 0).float()
        correct += (pred == sample['labels'].squeeze()).float().item()
rm_acc = correct / len(rm_val_tok)
print(f'\nRM val accuracy: {rm_acc:.3f} (random=0.5)')
rm_tracker.plot_ascii('Reward Model Training Loss')

In [ ]:
# ── Cell 7: Test RM on sample explanations ────────────────────────────────────
@torch.no_grad()
def get_reward_score(rm, tokenizer, prompt, response, max_len=128):
    """Get scalar reward from the trained RM."""
    text = prompt + ' ' + response
    enc  = tokenizer(text, truncation=True, max_length=max_len,
                     padding='max_length', return_tensors='pt')
    out  = rm(enc['input_ids'], enc['attention_mask'])
    return torch.sigmoid(out.logits.squeeze()).item()  # 0–1 score

sample_prompt = rm_val[0]['prompt']
good_response = rm_val[0]['response'] if rm_val[0]['label'] == 1 else rm_val[1]['response']
bad_response  = rm_val[1]['response'] if rm_val[1]['label'] == 0 else rm_val[0]['response']

score_good = get_reward_score(rm_model, rm_tokenizer, sample_prompt, good_response)
score_bad  = get_reward_score(rm_model, rm_tokenizer, sample_prompt, bad_response)

print('=== RM sanity check ===')
print(f'Good response score: {score_good:.3f}')
print(f'Bad response score:  {score_bad:.3f}')
print(f'RM correctly orders: {score_good > score_bad}')

In [ ]:
# ── Cell 8: Save RM ───────────────────────────────────────────────────────────
save_model(rm_model, rm_tokenizer, 'rlhf_reward_model')
print('Reward model saved.')

In [ ]:
# ── Cell 9: Phase 2 — Load policy for PPO ────────────────────────────────────
#
# PPO requires AutoModelForCausalLMWithValueHead — this adds a small
# value head on top of the LM to predict expected future reward (baseline).
# The value head is critical for variance reduction in PPO.
#
# Architecture:
#   Qwen backbone → LM head (next token)
#                 → value head (scalar V(s))

print('=== Phase 2: PPO fine-tuning ===')

sft_path = PROJECT_ROOT / 'models' / 'sft_cybersec'
policy_source = str(sft_path) if sft_path.exists() else MODEL_ID
print(f'Loading policy from: {policy_source}')

policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    policy_source,
    trust_remote_code=True,
    torch_dtype=torch.float32,
)
policy_model = policy_model.to('cpu')

from transformers import AutoTokenizer
policy_tokenizer = AutoTokenizer.from_pretrained(policy_source, trust_remote_code=True)
if policy_tokenizer.pad_token is None:
    policy_tokenizer.pad_token = policy_tokenizer.eos_token

print(f'Policy model loaded with value head.')

In [ ]:
# ── Cell 10: Configure PPO trainer ───────────────────────────────────────────
ppo_config = PPOConfig(
    # ── PPO hyperparams ──
    learning_rate=1e-5,
    batch_size=2,              # rollout buffer size (CPU: keep very small)
    mini_batch_size=1,
    ppo_epochs=1,              # PPO update epochs per rollout
    gamma=0.99,                # reward discount
    lam=0.95,                  # GAE lambda
    clip_range=0.2,            # PPO epsilon
    vf_coef=0.1,               # value loss coefficient
    # ── Generation ──
    target_kl=0.1,             # early stop if KL exceeds this
    # ── Misc ──
    log_with=None,
    remove_unused_columns=False,
)

ppo_trainer = PPOTrainer(
    model=policy_model,
    config=ppo_config,
    tokenizer=policy_tokenizer,
)

print('PPO trainer ready.')
print(f'  Rollout batch size: {ppo_config.batch_size}')
print(f'  PPO epochs/batch:   {ppo_config.ppo_epochs}')
print(f'  KL target:          {ppo_config.target_kl}')

In [ ]:
# ── Cell 11: PPO training loop ────────────────────────────────────────────────
#
# Manual PPO loop (more educational than using the auto-loop):
#   1. Tokenize prompt
#   2. Policy generates response (rollout)
#   3. RM scores the response → reward tensor
#   4. PPOTrainer.step() does the PPO update

GENERATION_KWARGS = {
    'max_new_tokens': 64,
    'do_sample': True,
    'temperature': 0.7,
    'pad_token_id': policy_tokenizer.pad_token_id,
    'eos_token_id': policy_tokenizer.eos_token_id,
}

N_PPO_STEPS = min(20, len(ppo_prompts) // ppo_config.batch_size)
ppo_tracker = LossTracker()

print(f'Starting PPO training: {N_PPO_STEPS} update steps...')
print('Each step: generate → score with RM → PPO update\n')

for step in range(N_PPO_STEPS):
    # ── 1. Sample a batch of prompts ──
    batch_prompts = ppo_prompts[step * ppo_config.batch_size : (step + 1) * ppo_config.batch_size]
    prompt_texts  = [p['prompt'] for p in batch_prompts]

    # ── 2. Tokenize prompts ──
    query_tensors = [
        policy_tokenizer(p, return_tensors='pt', truncation=True, max_length=64).input_ids.squeeze(0)
        for p in prompt_texts
    ]

    # ── 3. Generate responses (rollout) ──
    policy_model.eval()
    response_tensors = ppo_trainer.generate(query_tensors, **GENERATION_KWARGS)

    # Decode for reward computation
    response_texts = [
        policy_tokenizer.decode(r.squeeze(), skip_special_tokens=True)
        for r in response_tensors
    ]

    # ── 4. Score with RM ──
    rewards = [
        torch.tensor(get_reward_score(rm_model, rm_tokenizer, p, r))
        for p, r in zip(prompt_texts, response_texts)
    ]

    # ── 5. PPO update ──
    policy_model.train()
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    avg_reward = sum(r.item() for r in rewards) / len(rewards)
    ppo_loss   = stats.get('ppo/loss/total', 0)
    kl         = stats.get('ppo/mean_non_score_reward', 0)

    ppo_tracker.record(step, ppo_loss)

    if step % 5 == 0:
        print(f'  Step {step:3d} | avg_reward={avg_reward:.3f} | ppo_loss={ppo_loss:.4f} | kl={kl:.4f}')

print('\nPPO training complete.')
ppo_tracker.plot_ascii('PPO Training Loss')

In [ ]:
# ── Cell 12: Post-PPO qualitative evaluation ──────────────────────────────────
policy_model.eval()

test_prompt = (
    'Analyze the following URL and explain whether it is likely phishing or legitimate. '
    'Provide specific indicators.\n\n'
    'URL: http://paypa1-secure-login.xyz/verify?account=suspended\n\nAnalysis:'
)

# Generate using the base model interface
inputs = policy_tokenizer(test_prompt, return_tensors='pt', truncation=True, max_length=64)
with torch.no_grad():
    output = policy_model.generate(
        inputs['input_ids'],
        max_new_tokens=80, temperature=0.1, do_sample=False,
        pad_token_id=policy_tokenizer.pad_token_id
    )
new_tokens = output[0][inputs['input_ids'].shape[1]:]
response = policy_tokenizer.decode(new_tokens, skip_special_tokens=True)

print('=== PPO model response ===')
print(response)

# Score it with the RM
score = get_reward_score(rm_model, rm_tokenizer, test_prompt, response)
print(f'\nRM score: {score:.3f} (higher = RM thinks this is a better response)')

In [ ]:
# ── Cell 13: Save PPO model ───────────────────────────────────────────────────
# Save the pretrained part (without value head) for inference
rlhf_save_path = PROJECT_ROOT / 'models' / 'rlhf_policy'
rlhf_save_path.mkdir(parents=True, exist_ok=True)
policy_model.pretrained_model.save_pretrained(str(rlhf_save_path))
policy_tokenizer.save_pretrained(str(rlhf_save_path))

ppo_tracker.save(str(PROJECT_ROOT / 'models' / 'rlhf_ppo_loss_history.json'))
print(f'PPO policy saved to {rlhf_save_path}')
print('Notebook 4 complete.')

## Key takeaways

1. **RLHF is a two-model system**: the reward model and policy are separate. The RM is frozen during PPO; only the policy updates.

2. **The value head is essential**: without it, PPO has no baseline for variance reduction. The value head learns `V(s) ≈ E[future rewards from state s]`.

3. **KL penalty prevents reward hacking**: without `β * KL(π_θ ∥ π_ref)`, the policy quickly learns to generate adversarial completions that score high on the RM but are nonsensical.

4. **Why DPO often wins in practice**: DPO achieves similar alignment with one-tenth the complexity. RLHF's advantage shows when you need online learning — the policy generating its own training data in a feedback loop.

5. **Reward hacking example**: if our RM learned that longer responses score higher, PPO would generate extremely verbose responses. Always audit your RM before deploying PPO at scale.

## Full pipeline summary
```
Base Qwen2.5-0.5B
    │
    ▼ SFT (01_sft.ipynb)    — imitation learning on expert Q&A
    │
    ▼ DPO (02_dpo.ipynb)    — preference alignment without RL
    │
    ▼ GRPO (03_grpo.ipynb)  — verifiable reward, group-relative update
    │
    ▼ RLHF (04_rlhf.ipynb)  — learned reward model + PPO online update
    │
    ▼ Aligned cybersecurity assistant
```